# 04 · Climate drivers and CH2025 projections of larch autumn phenology

1. **Which climate factors influence larch autumn phenology, and how?**
   - Correlation scan over time windows
   - Effects per climate variable (days per °C, per 100 mm, …)
   - A process-based senescence model
   - A random-forest cross-check
2. **Do the CH2025 climate scenarios give better predictions of future yellowing?**
   - Leave-one-year-out cross-validation against a mean-only model and the linear trend of `03`
   - The best models projected with all CH2025 model chains and warming levels

The phenology date to analyse is selectable: `TARGETS` in `config.py`, either a chosen date of `03` (`"start"`, `"peak"`, `"end"`) or one candidate of `SEASON_METHODS` (e.g. `"Whittaker / GRVI / mid"`). Every step runs for each target.

**Climate sources**, used side by side:
- **grid:** gridded observations, projected with CH2025 DAILY-GRIDDED
- **station:** Buffalora observations, projected with CH2025 DAILY-LOCAL

A source whose files are missing is skipped.

- **Reads:**
  - `results/<AOI>/season_selected.csv` and `season_dates.csv` (from `03`)
  - `cache/climate/<AOI>/*_zones.nc` (from `02`)
- **Writes:** `results/<AOI>/climate/`

**Sample size.** Climate varies mainly from year to year, and all zones share the same year, so the effective sample is about the number of years (≈ 10), not the number of zone-years. All uncertainty estimates account for this:
- CIs are clustered by year;
- cross-validation leaves out whole years;
- permutation tests shuffle whole years.

Treat effects as indicative.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Transformer
from scipy.optimize import differential_evolution
from scipy.stats import t as t_dist
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import load_zone_table

CLIMATE_FILES = {   # source: (observations, scenarios)
    "grid": ("obs_grid_zones.nc", "ch2025_grid_zones.nc"),
    "station": ("obs_station_zones.nc", "ch2025_local_zones.nc"),
}
SCREEN_VARS = ["tas", "tasmin", "pr"]           # daily variables of the window scan
SCREEN_STARTS = list(range(121, 289, 7))        # window start (DOY, 1 May - mid Oct)
SCREEN_LENGTHS = [10, 20, 30, 45, 60, 90]       # window length (days)
N_PERMUTATIONS = 500
MAX_FEATURES = 2                                # climate variables per regression model
PROCESS_VARIANTS = {"cold degree-days": (1, 0), "Delpierre": (1, 1)}   # exponents (x, y), see section 3c
REFERENCE = "ref91-20"
# Approximate time when the global warming levels are reached (IPCC AR6 WG1, central estimates)
GWL_TIMING = {"ref91-20": "1991-2020", "gwl1.5": "~2030s", "gwl2.0": "~2040s-2060s",
              "gwl2.5": "~2050s-2080s", "gwl3.0": "~2060s-2100 (high emissions only)"}

CLIMATE_OUT_DIR.mkdir(parents=True, exist_ok=True)
zone_table = load_zone_table()
ZONES = [z for z in zone_table.index if z != "all"]
ZONE_SHARE = zone_table.loc[ZONES, "n_pixels"] / zone_table.loc[ZONES, "n_pixels"].sum()   # for AOI means
DOYS = np.arange(1, 366)
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | targets: {TARGETS}")

## 1. Phenology dates (targets)

`"start"`, `"peak"` and `"end"` are the chosen dates of `03` (`SEASON_CHOICE`, complete years only); a candidate such as `"Whittaker / GRVI / mid"` is taken from `season_dates.csv`, without the preliminary years. Models use the topographic zones; `"all"` is only shown for reference.

In [ ]:
def load_targets():
    # "start" / "peak" / "end": the chosen dates of 03; "<method> / <index> / <metric>": one candidate of 03
    selected = pd.read_csv(OUT_DIR / "season_selected.csv")
    dates = pd.read_csv(OUT_DIR / "season_dates.csv")
    dates = dates[~dates.preliminary]
    parts = []
    for target in TARGETS:
        if target in ("start", "peak", "end"):
            d = selected.rename(columns={target: "doy"})
        else:
            key = [part.strip() for part in target.split("/")]
            d = (dates[(dates.method == key[0]) & (dates["index"] == key[1]) & (dates.metric == key[2])]
                 if len(key) == 3 else dates.iloc[:0])
            if d.empty:
                options = sorted(set(dates.method + " / " + dates["index"] + " / " + dates.metric))
                raise ValueError(f"TARGETS: {target!r} is not in season_dates.csv. Options: start, peak, end, {options}")
        parts.append(d[["zone", "year", "doy"]].dropna().assign(target=target))
    out = pd.concat(parts, ignore_index=True)
    return out[out.year.between(FIRST_YEAR, LAST_YEAR)]


targets = load_targets()
display(targets[targets.zone != "all"].pivot_table(index="target", columns="year", values="doy", aggfunc="size")
        .fillna(0).astype(int).rename_axis(columns="zones per year"))

fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 3.5), sharey=True, squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    d = targets[targets.target == target]
    for zone, g in d[d.zone != "all"].groupby("zone"):
        ax.plot(g.year, g.doy, "-", lw=0.8, alpha=0.5)
    a = d[d.zone == "all"]
    ax.plot(a.year, a.doy, "ko-", lw=2, label="whole AOI")
    ax.set_title(target)
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
axes[0, 0].legend()
plt.suptitle("Phenology dates per zone (thin) and for the whole AOI")
plt.tight_layout()
plt.show()

## 2. Climate data and features per zone and year

The daily zone series of `02` are arranged as (zone, year, day of year) with 365 days; 29 February is dropped. Model years of CH2025 are numbered 1–30 per warming level.

The **features** are seasonal summaries of each year. They are computed the same way for observations and scenarios, and only from the variables a source has. The windows end by mid October so that they precede most transition dates. Snow is not in the scenarios; winter precipitation serves as a snowpack proxy.

In [ ]:
def to_matrix(ds):
    # (..., time) -> (..., year, doy); 29 Feb dropped so every year has 365 days
    leap_day = (ds.month == 2) & (ds.doy == 60)
    ds = ds.isel(time=np.flatnonzero(~leap_day.values)).drop_vars("month")
    return ds.set_index(time=["year", "doy"]).unstack("time").reindex(doy=DOYS)


climate = {}
for source, (obs_file, scen_file) in CLIMATE_FILES.items():
    if not (CLIMATE_DIR / obs_file).exists():
        print(f"{source}: {obs_file} missing (run 02_download_climate), source skipped")
        continue
    obs = to_matrix(xr.open_dataset(CLIMATE_DIR / obs_file).load())
    scen = None
    if (CLIMATE_DIR / scen_file).exists():
        scen = to_matrix(xr.open_dataset(CLIMATE_DIR / scen_file).load())
    else:
        print(f"{source}: {scen_file} missing, no projections for this source")
    climate[source] = {"obs": obs.sel(zone=ZONES), "scen": None if scen is None else scen.sel(zone=ZONES)}
    print(f"{source}: observations {list(obs.data_vars)}, years {int(obs.year.min())}-{int(obs.year.max())}"
          + ("" if scen is None else f" | scenarios {dict(scen.sizes)}"))
if not climate:
    raise FileNotFoundError(f"No climate data in {CLIMATE_DIR}; run 02_download_climate first")

cells = pd.read_csv(CLIMATE_DIR / "zone_cells.csv")
_, LATITUDE = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True).transform(cells.E.mean(), cells.N.mean())

In [ ]:
def daylength(lat, doy):
    # Hours of daylight (CBM model, Forsythe et al. 1995, sunrise/-set at the top of the solar disc)
    theta = 0.2163108 + 2 * np.arctan(0.9671396 * np.tan(0.00860 * (doy - 186)))
    phi = np.arcsin(0.39795 * np.cos(theta))
    arg = (np.sin(np.radians(0.8333)) + np.sin(np.radians(lat)) * np.sin(phi)) / (np.cos(np.radians(lat)) * np.cos(phi))
    return 24 - 24 / np.pi * np.arccos(np.clip(arg, -1, 1))


def extraterrestrial_radiation(lat, doy):
    # FAO-56 eq. 21, MJ m-2 day-1
    phi = np.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(-np.tan(phi) * np.tan(delta))
    return 24 * 60 / np.pi * 0.0820 * dr * (ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))


PHOTOPERIOD = daylength(LATITUDE, DOYS)
RA = xr.DataArray(extraterrestrial_radiation(LATITUDE, DOYS), dims="doy", coords={"doy": DOYS})


def season(da, first, last, how="mean"):
    # Mean or sum over DOY first..last of every year; NaN if a day is missing
    win = da.sel(doy=slice(first, last))
    return win.mean("doy", skipna=False) if how == "mean" else win.sum("doy", skipna=False)


def winter_sum(da):
    # 1 Nov of the previous year - 30 Apr
    return season(da, 305, 365, "sum").shift(year=1) + season(da, 1, 120, "sum")


def pet_hargreaves(ds):
    # mm/day (Hargreaves & Samani 1985)
    return 0.0023 * 0.408 * RA * (ds.tas + 17.8) * np.sqrt((ds.tasmax - ds.tasmin).clip(min=0))


def vpd(ds):
    # Vapour pressure deficit (hPa) from mean temperature and relative humidity
    return 6.112 * np.exp(17.62 * ds.tas / (243.12 + ds.tas)) * (1 - ds.hurs / 100)


# name: (description, unit, required variables, function of the daily dataset)
FEATURES = {
    "T_spring": ("mean temperature Mar-May", "°C", ["tas"], lambda d: season(d.tas, 60, 151)),
    "GDD5_JanJul": ("growing degree-days above 5 °C, Jan-Jul", "°C d", ["tas"],
                    lambda d: season((d.tas - 5).clip(min=0), 1, 212, "sum")),
    "T_summer": ("mean temperature Jun-Aug", "°C", ["tas"], lambda d: season(d.tas, 152, 243)),
    "T_Sep": ("mean temperature Sep", "°C", ["tas"], lambda d: season(d.tas, 244, 273)),
    "T_Oct1": ("mean temperature 1-15 Oct", "°C", ["tas"], lambda d: season(d.tas, 274, 288)),
    "Tmin_Sep": ("mean minimum temperature Sep", "°C", ["tasmin"], lambda d: season(d.tasmin, 244, 273)),
    "frost_days": ("days with Tmin < 0 °C, 15 Aug-15 Oct", "days", ["tasmin"],
                   lambda d: season((d.tasmin < 0).where(d.tasmin.notnull()), 227, 288, "sum")),
    "CDD10_AugSep": ("cold degree-days below 10 °C, Aug-Sep", "°C d", ["tas"],
                     lambda d: season((10 - d.tas).clip(min=0), 213, 273, "sum")),
    "P_winter": ("precipitation Nov-Apr (snowpack proxy)", "mm", ["pr"], lambda d: winter_sum(d.pr)),
    "P_summer": ("precipitation Jun-Aug", "mm", ["pr"], lambda d: season(d.pr, 152, 243, "sum")),
    "P_AugSep": ("precipitation Aug-Sep", "mm", ["pr"], lambda d: season(d.pr, 213, 273, "sum")),
    "CWB_summer": ("climatic water balance P - PET (Hargreaves), Jun-Sep", "mm", ["pr", "tas", "tasmin", "tasmax"],
                   lambda d: season(d.pr - pet_hargreaves(d), 152, 273, "sum")),
    "sunshine_AugSep": ("relative sunshine duration Aug-Sep", "%", ["srel"], lambda d: season(d.srel, 213, 273)),
    "radiation_AugSep": ("global radiation Aug-Sep", "W/m²", ["rsds"], lambda d: season(d.rsds, 213, 273)),
    "VPD_summer": ("vapour pressure deficit Jul-Sep", "hPa", ["tas", "hurs"], lambda d: season(vpd(d), 182, 273)),
    "ET0_summer": ("FAO reference evaporation Jun-Sep", "mm", ["et0"], lambda d: season(d.et0, 152, 273, "sum")),
}


def compute_features(ds):
    return xr.Dataset({name: f(ds) for name, (_, _, needs, f) in FEATURES.items() if set(needs) <= set(ds.data_vars)})


info_rows, obs_features = [], {}
for source, c in climate.items():
    feats = compute_features(c["obs"])
    obs_features[source] = feats.to_dataframe().reset_index()
    c["features_scen"] = None if c["scen"] is None else compute_features(c["scen"])
    for name in feats.data_vars:
        info_rows.append({"source": source, "feature": name, "description": FEATURES[name][0],
                          "unit": FEATURES[name][1],
                          "in_scenarios": c["features_scen"] is not None and name in c["features_scen"]})
feature_info = pd.DataFrame(info_rows)
display(feature_info.pivot_table(index=["feature", "description", "unit"], columns="source", values="in_scenarios",
                                 aggfunc="first").replace({True: "obs + scenarios", False: "obs only"}))


def model_data(target, source):
    # One row per zone-year: phenology date + features
    d = targets[(targets.target == target) & targets.zone.isin(ZONES)]
    return d.merge(obs_features[source], on=["zone", "year"], how="inner").reset_index(drop=True)

## 3. Q1: Which climate factors matter, and how?

### 3a. Window scan

For each daily variable, the mean over a window (start day × length) is correlated with the phenology date.

- **Within zones:** both are centred on each zone's mean, so elevation differences between zones cannot pose as climate effects. What remains is year-to-year variation.
- **Windows end before the median date** of the target.
- **Significance:** the years are shuffled `N_PERMUTATIONS` times, keeping all zones of a year together. The dashed contour marks windows whose |r| exceeds the 95th percentile of the largest |r| under shuffling, i.e. significant even after scanning all windows.

Blue: warmer / wetter → earlier date. Red: → later date.

In [ ]:
def window_means(da, starts, lengths, min_share=0.9):
    # (zone, year, doy) -> (start, length, zone, year) window means; NaN if < min_share of the days exist
    x = da.transpose("zone", "year", "doy").values
    valid = np.isfinite(x)
    pad = np.zeros(x.shape[:2] + (1,))
    csum = np.concatenate([pad, np.cumsum(np.where(valid, x, 0), axis=2)], axis=2)
    cnt = np.concatenate([pad, np.cumsum(valid, axis=2)], axis=2)
    out = np.full((len(starts), len(lengths)) + x.shape[:2], np.nan)
    for i, s in enumerate(starts):
        for j, n in enumerate(lengths):
            e = s + n - 1
            if e <= 365:
                k = cnt[..., e] - cnt[..., s - 1]
                out[i, j] = np.where(k >= min_share * n, (csum[..., e] - csum[..., s - 1]) / np.maximum(k, 1), np.nan)
    return out


def within_r(x, y):
    # Correlation of x (..., zone, year) with y (zone, year) after removing each zone's mean
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = np.where(ok, x, np.nan), np.where(ok, y, np.nan)
    with np.errstate(invalid="ignore", divide="ignore"), np.testing.suppress_warnings() as sup:
        sup.filter(RuntimeWarning)
        x = x - np.nanmean(x, axis=-1, keepdims=True)
        y = y - np.nanmean(y, axis=-1, keepdims=True)
        return np.nansum(x * y, axis=(-2, -1)) / np.sqrt(np.nansum(x * x, axis=(-2, -1)) * np.nansum(y * y, axis=(-2, -1)))


rng = np.random.default_rng(0)
scan_source = "grid" if "grid" in climate else next(iter(climate))
obs = climate[scan_source]["obs"]
scan_vars = [v for v in SCREEN_VARS if v in obs]
windows = {v: window_means(obs[v], SCREEN_STARTS, SCREEN_LENGTHS) for v in scan_vars}
starts, lengths = np.array(SCREEN_STARTS)[:, None], np.array(SCREEN_LENGTHS)[None, :]

scan_rows = []
fig, axes = plt.subplots(len(TARGETS), len(scan_vars), figsize=(4.5 * len(scan_vars), 3 * len(TARGETS)),
                         squeeze=False, sharex=True, sharey=True)
for i, target in enumerate(TARGETS):
    d = targets[(targets.target == target) & targets.zone.isin(ZONES)]
    y = (d.pivot_table(index="zone", columns="year", values="doy")
         .reindex(index=obs.zone.values, columns=obs.year.values).values)
    before = (starts + lengths - 1) <= np.nanmedian(y)        # window ends before the median date
    r = {v: np.where(before, within_r(w, y), np.nan) for v, w in windows.items()}
    perm_max = []
    for _ in range(N_PERMUTATIONS):
        yp = y[:, rng.permutation(y.shape[1])]
        perm_max.append(max(np.nanmax(np.abs(np.where(before, within_r(w, yp), np.nan))) for w in windows.values()))
    r_crit = np.quantile(perm_max, 0.95)
    for j, v in enumerate(scan_vars):
        ax = axes[i, j]
        im = ax.pcolormesh(SCREEN_STARTS, SCREEN_LENGTHS, r[v].T, cmap="RdBu_r", vmin=-1, vmax=1, shading="nearest")
        ax.contour(SCREEN_STARTS, SCREEN_LENGTHS, (np.abs(r[v]) >= r_crit).T.astype(float), levels=[0.5],
                   colors="k", linestyles="--", linewidths=1)
        ax.set_title(f"{target} vs {v}", fontsize=10)
        k = np.unravel_index(np.nanargmax(np.abs(r[v])), r[v].shape)
        s, n = SCREEN_STARTS[k[0]], SCREEN_LENGTHS[k[1]]
        scan_rows.append({"target": target, "variable": v, "best_start_doy": s, "best_end_doy": s + n - 1,
                          "best_window": f"{pd.Timestamp(2023, 1, 1) + pd.Timedelta(days=s - 1):%d %b}-"
                                         f"{pd.Timestamp(2023, 1, 1) + pd.Timedelta(days=s + n - 2):%d %b}",
                          "r_within": r[v][k], "r_crit_95": r_crit, "significant": abs(r[v][k]) >= r_crit})
for ax in axes[-1]:
    ax.set_xlabel("window start (DOY)")
for ax in axes[:, 0]:
    ax.set_ylabel("length (days)")
fig.colorbar(im, ax=axes, label="within-zone r", shrink=0.6)
plt.suptitle(f"Window scan ({scan_source} observations); dashed: significant after scanning all windows")
plt.show()
window_scan = pd.DataFrame(scan_rows)
display(window_scan.round(3))

### 3b. Effect of each climate feature

For each target and feature, an OLS model with zone fixed effects is fitted: `doy ~ feature + C(zone)`.

- **`slope`:** the change in date per unit of the feature, estimated from year-to-year variation within zones. CIs are clustered by year, using t quantiles with (years − 1) degrees of freedom.
- **`r2_within`:** the share of within-zone variation explained.
- **`space_for_time`:** the slope across zone means (elevation gradient), for comparison. If it is similar to the temporal slope, warmer sites behave like warmer years. If not, other site factors play a role.

In [ ]:
def demean(d, cols):
    return d[cols] - d.groupby("zone")[cols].transform("mean")


def feature_effect(d, feature):
    d = d.dropna(subset=[feature, "doy"])
    n_years = d.year.nunique()
    if n_years < 4:
        return None
    x, y = demean(d, [feature])[feature].values, demean(d, ["doy"])["doy"].values
    if np.allclose(x, 0):
        return None
    slope = (x @ y) / (x @ x)
    resid = y - slope * x
    # Cluster-robust (by year) standard error of the slope
    score = pd.Series(x * resid).groupby(d.year.values).sum().values
    se = np.sqrt(n_years / (n_years - 1) * (score @ score)) / (x @ x)
    q = t_dist.ppf(1 - ALPHA / 2, n_years - 1)
    zone_means = d.groupby("zone")[[feature, "doy"]].mean()
    space = np.polyfit(zone_means[feature], zone_means["doy"], 1)[0] if len(zone_means) >= 3 else np.nan
    return {"slope": slope, "ci_low": slope - q * se, "ci_high": slope + q * se,
            "p": 2 * t_dist.sf(abs(slope / se), n_years - 1), "r2_within": 1 - (resid @ resid) / (y @ y),
            "space_for_time": space, "n_zone_years": len(d), "n_years": n_years}


effect_rows = []
for source in climate:
    for target in TARGETS:
        d = model_data(target, source)
        for feature in obs_features[source].columns.drop(["zone", "year"]):
            e = feature_effect(d, feature)
            if e:
                effect_rows.append({"source": source, "target": target, "feature": feature,
                                    "unit": f"days per {FEATURES[feature][1]}", **e})
climate_effects = pd.DataFrame(effect_rows)
for (source, target), g in climate_effects.groupby(["source", "target"], sort=False):
    print(f"{target} ({source} observations)")
    display(g.drop(columns=["source", "target"]).sort_values("p").set_index("feature").round(3))

In [ ]:
# Partial plots of the three strongest features per target (within-zone deviations, coloured by year)
src = scan_source
for target in TARGETS:
    d = model_data(target, src)
    top = climate_effects[(climate_effects.source == src) & (climate_effects.target == target)].nsmallest(3, "p")
    if top.empty:
        continue
    fig, axes = plt.subplots(1, len(top), figsize=(4.2 * len(top), 3.6), squeeze=False)
    for ax, row in zip(axes[0], top.itertuples()):
        dd = d.dropna(subset=[row.feature])
        x, y = demean(dd, [row.feature])[row.feature], demean(dd, ["doy"])["doy"]
        sc = ax.scatter(x, y, c=dd.year, cmap="viridis", s=18)
        xx = np.array([x.min(), x.max()])
        ax.plot(xx, row.slope * xx, "k-")
        ax.set_xlabel(f"{row.feature} deviation ({FEATURES[row.feature][1]})")
        ax.set_title(f"{row.slope:+.2f} d per {FEATURES[row.feature][1]} (p={row.p:.3f})", fontsize=10)
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel("date deviation (days)")
    fig.colorbar(sc, ax=axes, label="year")
    plt.suptitle(f"{target}: strongest climate effects ({src} observations)")
    plt.show()

### 3c. Process-based senescence model

This is the senescence model of Delpierre et al. (2009), widely used for the autumn colouring of deciduous trees and larch.

- Once the day length falls below `P_start`, every day colder than `T_b` adds `(T_b − T)^x · (P / P_start)^y` to a "cold sum".
- Senescence is reached when the sum exceeds `Y_crit`.

Two variants:
- **cold degree-days:** `x = 1`, `y = 0`;
- **Delpierre:** `x = 1`, `y = 1`. Shorter days weight the cold more.

The parameters are fitted with differential evolution to the zone-years, using each zone's own daily temperature. This gives a mechanistic reading: which base temperature and start day length explain the dates. Because day length caps how late senescence can occur, it extrapolates more safely than a linear regression.

In [ ]:
def process_doy(temp, params, x, y):
    # temp (..., 365) daily mean temperature -> DOY when the cold sum reaches Y_crit (NaN if never)
    t_base, p_start, y_crit = params
    start = np.argmax((DOYS > 172) & (PHOTOPERIOD < p_start))   # first day after the solstice below P_start
    rate = np.where(temp < t_base, (t_base - temp) ** x, 0.0) * (PHOTOPERIOD / p_start) ** y
    rate[..., :start] = 0
    hit = np.cumsum(np.nan_to_num(rate), axis=-1) >= y_crit
    return np.where(hit.any(-1), hit.argmax(-1) + 1.0, np.nan)


PROCESS_BOUNDS = [(0, 25), (10.5, 15.5), (1, 3000)]   # T_b (°C), P_start (h), Y_crit


def fit_process(temp, doy, x, y, maxiter=150):
    def loss(p):
        err = process_doy(temp, p, x, y) - doy
        return np.sqrt(np.mean(np.where(np.isnan(err), 60.0, err) ** 2))   # no senescence = 60 days off
    res = differential_evolution(loss, PROCESS_BOUNDS, seed=0, maxiter=maxiter, polish=False, tol=1e-8)
    return res.x, res.fun


def zone_year_temp(obs, d):
    # Daily temperature rows (len(d), 365) for the zone-years of d
    t = obs.tas.sel(zone=xr.DataArray(d.zone.values, dims="row"), year=xr.DataArray(d.year.values, dims="row"))
    return t.transpose("row", "doy").values


process_rows = []
for source, c in climate.items():
    if "tas" not in c["obs"]:
        continue
    for target in TARGETS:
        d = model_data(target, source)
        d = d[d.year.isin(c["obs"].year.values)]
        temp = zone_year_temp(c["obs"], d)
        complete = np.isfinite(temp[:, 150:]).all(1)          # summer-autumn temperature available
        d, temp = d[complete], temp[complete]
        for name, (x, y) in PROCESS_VARIANTS.items():
            params, rmse = fit_process(temp, d.doy.values, x, y)
            pred = process_doy(temp, params, x, y)
            process_rows.append({"source": source, "target": target, "variant": name, "T_base": params[0],
                                 "P_start_h": params[1], "Y_crit": params[2], "rmse": rmse,
                                 "r": pd.Series(pred).corr(pd.Series(d.doy.values)),
                                 "start_doy": int(np.argmax((DOYS > 172) & (PHOTOPERIOD < params[1]))) + 1,
                                 "n_zone_years": len(d)})
process_params = pd.DataFrame(process_rows)
display(process_params.round(2))

### 3d. Cross-check: random forest

A random forest uses all features of a source plus zone elevation. Permutation importance is computed on held-out years (leave-one-year-out), so it reflects predictive value rather than fit to the training data. With so few years, this is only a plausibility check of 3a–3c.

In [ ]:
rf_rows = []
for source in climate:
    for target in TARGETS:
        d = model_data(target, source).join(zone_table.elev_mean, on="zone")
        feats = [f for f in obs_features[source].columns.drop(["zone", "year"]) if d[f].notna().mean() > 0.8]
        d = d.dropna(subset=feats)
        if d.year.nunique() < 4:
            continue
        cols = feats + ["elev_mean"]
        imp = []
        for year in d.year.unique():
            tr, te = d[d.year != year], d[d.year == year]
            rf = RandomForestRegressor(300, min_samples_leaf=3, random_state=0).fit(tr[cols], tr.doy)
            imp.append(permutation_importance(rf, te[cols], te.doy, n_repeats=10, random_state=0,
                                              scoring="neg_mean_absolute_error").importances_mean)
        rf_rows += [{"source": source, "target": target, "feature": f, "importance_days": v}
                    for f, v in zip(cols, np.mean(imp, axis=0))]
rf_importance = pd.DataFrame(rf_rows)
display(rf_importance.pivot_table(index="feature", columns=["source", "target"], values="importance_days").round(2))

## 4. Q2: Do climate models predict better than the trend?

### 4a. Leave-one-year-out cross-validation

Each year is predicted by models fitted on the other years only.

| Model | How it predicts |
|---|---|
| mean | each zone's mean date |
| trend | zone means + a common linear trend in years (the baseline of `03`) |
| climate regression | zone means + up to `MAX_FEATURES` climate features |
| process models (3c) | the cold-sum models, refitted in each fold |

- The climate regression chooses its features by forward selection with an inner leave-one-year-out loop. The choice is part of each fold, so the scores are not optimistic.
- Only features that also exist in the scenarios of that source are candidates, because these models are used for projection.
- **`skill`** = 1 − RMSE / RMSE(mean). Above 0 means better than the mean-only model.

In [ ]:
def fe_fit(d, feats):
    # OLS with zone fixed effects: doy ~ feats + zone
    zones = pd.get_dummies(d.zone, dtype=float)
    X = np.column_stack([d[feats].values, zones.values]) if feats else zones.values
    beta = np.linalg.lstsq(X, d.doy.values, rcond=None)[0]
    return {"features": list(feats), "slopes": beta[:len(feats)], "zones": pd.Series(beta[len(feats):], zones.columns)}


def fe_predict(model, d):
    base = d.zone.map(model["zones"]).values
    return base + (d[model["features"]].values @ model["slopes"] if model["features"] else 0)


def loyo_rmse(d, feats):
    err = []
    for year in d.year.unique():
        tr, te = d[d.year != year], d[d.year == year]
        err.append(fe_predict(fe_fit(tr, feats), te) - te.doy.values)
    err = np.concatenate(err)
    return np.sqrt(np.nanmean(err ** 2))


def select_features(d, candidates):
    chosen, best = [], loyo_rmse(d, [])
    while len(chosen) < MAX_FEATURES:
        scores = {f: loyo_rmse(d, chosen + [f]) for f in candidates if f not in chosen}
        if not scores:
            break
        f, s = min(scores.items(), key=lambda kv: kv[1])
        if s >= best:
            break
        chosen.append(f)
        best = s
    return chosen


def scores(pred, obs):
    err = pred - obs
    return {"rmse": np.sqrt(np.nanmean(err ** 2)), "mae": np.nanmean(np.abs(err)), "bias": np.nanmean(err),
            "r": pd.Series(pred).corr(pd.Series(obs)), "n": int(np.isfinite(err).sum())}


cv_rows, chosen_rows, final_models = [], [], {}
for source, c in climate.items():
    if c["features_scen"] is None:
        print(f"{source}: no scenarios, cross-validation skipped")
        continue
    candidates = [f for f in c["features_scen"].data_vars if f in obs_features[source]]
    for target in TARGETS:
        d = model_data(target, source)
        candidates_t = [f for f in candidates if d[f].notna().mean() > 0.8]
        d = d.dropna(subset=candidates_t).reset_index(drop=True)
        d = d[d.year.isin(c["obs"].year.values)].reset_index(drop=True)
        temp = zone_year_temp(c["obs"], d)
        if d.year.nunique() < 4:
            print(f"{target} ({source}): fewer than 4 years, skipped")
            continue
        pred = {name: np.full(len(d), np.nan) for name in ["mean", "trend", "climate regression", *PROCESS_VARIANTS]}
        for year in d.year.unique():
            tr, te = d.year != year, d.year == year
            pred["mean"][te] = fe_predict(fe_fit(d[tr], []), d[te])
            pred["trend"][te] = fe_predict(fe_fit(d[tr], ["year"]), d[te])
            feats = select_features(d[tr], candidates_t)
            chosen_rows.append({"source": source, "target": target, "left_out": year, "features": ", ".join(feats)})
            pred["climate regression"][te] = fe_predict(fe_fit(d[tr], feats), d[te])
            for name, (x, y) in PROCESS_VARIANTS.items():
                params, _ = fit_process(temp[tr.values], d.doy[tr].values, x, y, maxiter=80)
                pred[name][te] = process_doy(temp[te.values], params, x, y)
        for name, p in pred.items():
            cv_rows.append({"source": source, "target": target, "model": name, **scores(p, d.doy.values)})
        # Final models on all years, used for the projections
        feats = select_features(d, candidates_t)
        final_models[(source, target)] = {
            "data": d,
            "climate regression": fe_fit(d, feats),
            **{name: fit_process(temp, d.doy.values, x, y)[0] for name, (x, y) in PROCESS_VARIANTS.items()},
        }

cv_scores = pd.DataFrame(cv_rows)
cv_scores["skill"] = 1 - cv_scores.rmse / cv_scores.groupby(["source", "target"]).rmse.transform(
    lambda s: s[cv_scores.loc[s.index, "model"] == "mean"].iloc[0])
display(cv_scores.set_index(["source", "target", "model"]).round(2))
chosen = pd.DataFrame(chosen_rows)
display(chosen.groupby(["source", "target"]).features.value_counts().rename("folds").to_frame())

fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 3.5), squeeze=False, sharey=True)
for ax, target in zip(axes[0], TARGETS):
    s = cv_scores[cv_scores.target == target].pivot(index="model", columns="source", values="rmse")
    s.plot.barh(ax=ax, legend=ax is axes[0, -1])
    ax.set_title(target)
    ax.set_xlabel("RMSE (days), left-out years")
    ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

### 4b. Projections with CH2025

The final models, fitted on all years, are applied to every CH2025 model chain, warming level and model year. CH2025 is bias-adjusted to the observations, so the predicted dates are absolute dates.

Two summaries are reported per zone, warming level and model:
- **`doy_*`:** the 30-year mean date of each model chain. `doy_median` is the median over chains; `doy_p05` / `doy_p95` are the 5th and 95th percentiles.
- **`delta_*`:** the shift relative to `ref91-20` of the same chain. This removes the remaining model bias.

The whole AOI is the pixel-weighted mean of the zones.

**Extrapolation:** `extrapolation` gives, per selected feature and warming level, the share of scenario zone-years outside the observed range. A regression is a straight-line guess there. The process models are bounded by day length.

In [ ]:
def regression_projection(model, feats_scen):
    # (period, model, zone, year) predicted DOY
    zone_eff = xr.DataArray(model["zones"].reindex(ZONES).values, dims="zone", coords={"zone": ZONES})
    out = zone_eff + 0 * feats_scen[list(feats_scen.data_vars)[0]]
    for f, b in zip(model["features"], model["slopes"]):
        out = out + b * feats_scen[f]
    return out


def process_projection(params, scen, x, y):
    t = scen.tas.transpose(..., "doy")
    return xr.DataArray(process_doy(t.values, params, x, y), dims=t.dims[:-1], coords={k: t[k] for k in t.dims[:-1]})


proj_rows, extrap_rows, chain_means = [], [], []
for (source, target), fm in final_models.items():
    c = climate[source]
    projections = {"climate regression": regression_projection(fm["climate regression"], c["features_scen"])}
    for name, (x, y) in PROCESS_VARIANTS.items():
        projections[name] = process_projection(fm[name], c["scen"], x, y)
    obs_mean = (fm["data"].groupby("zone").doy.mean() * ZONE_SHARE).sum() / ZONE_SHARE[fm["data"].zone.unique()].sum()
    for model_name, p in projections.items():
        per_chain = p.mean("year")                                    # 30-year mean per chain
        per_chain = xr.concat([per_chain, (per_chain * xr.DataArray(ZONE_SHARE)).sum("zone", min_count=1)
                               .expand_dims(zone=["AOI"])], "zone")
        delta = per_chain - per_chain.sel(period=REFERENCE)
        for period in per_chain.period.values:
            for zone in per_chain.zone.values:
                v = per_chain.sel(period=period, zone=zone).values
                dv = delta.sel(period=period, zone=zone).values
                if np.isfinite(v).sum() == 0:
                    continue
                proj_rows.append({
                    "source": source, "target": target, "model": model_name, "period": period,
                    "timing": GWL_TIMING.get(str(period), ""), "zone": zone,
                    "doy_median": np.nanmedian(v), "doy_p05": np.nanpercentile(v, 5), "doy_p95": np.nanpercentile(v, 95),
                    "delta_median": np.nanmedian(dv), "delta_p05": np.nanpercentile(dv, 5),
                    "delta_p95": np.nanpercentile(dv, 95), "n_chains": int(np.isfinite(v).sum()),
                    "observed_mean_doy": obs_mean if zone == "AOI" else fm["data"].loc[fm["data"].zone == zone, "doy"].mean(),
                })
        chain_means.append(per_chain.sel(zone="AOI").to_dataframe(name="doy").reset_index()
                           .assign(source=source, target=target, model=model_name))
    for f in fm["climate regression"]["features"]:
        lo, hi = fm["data"][f].min(), fm["data"][f].max()
        sv = c["features_scen"][f]
        outside = ((sv < lo) | (sv > hi)).where(sv.notnull()).mean(["model", "zone", "year"])
        for period, share in outside.to_series().items():
            extrap_rows.append({"source": source, "target": target, "feature": f, "period": period,
                                "share_outside_observed_range": share})

projections_by_gwl = pd.DataFrame(proj_rows)
extrapolation = pd.DataFrame(extrap_rows)
display(projections_by_gwl[projections_by_gwl.zone == "AOI"].set_index(["source", "target", "model", "period"])
        .drop(columns="zone").round(1))
if not extrapolation.empty:
    display(extrapolation.pivot_table(index=["source", "target", "feature"], columns="period",
                                      values="share_outside_observed_range").round(2))

In [ ]:
if chain_means:
    chains = pd.concat(chain_means)
    periods = [p for p in CH2025_PERIODS if p in set(chains.period)]
    for target in TARGETS:
        ct = chains[chains.target == target]
        if ct.empty:
            continue
        combos = ct.groupby(["source", "model"]).size().index
        fig, ax = plt.subplots(figsize=(1.6 * len(periods) * len(combos) ** 0.5 + 4, 4.5))
        width = 0.8 / len(combos)
        for k, (source, model_name) in enumerate(combos):
            g = ct[(ct.source == source) & (ct.model == model_name)]
            data = [g[g.period == p].doy.dropna() for p in periods]
            pos = np.arange(len(periods)) + (k - (len(combos) - 1) / 2) * width
            bp = ax.boxplot(data, positions=pos, widths=width * 0.9, patch_artist=True, showfliers=False)
            for patch in bp["boxes"]:
                patch.set_facecolor(plt.get_cmap("tab10")(k))
                patch.set_alpha(0.6)
            ax.plot([], [], "s", color=plt.get_cmap("tab10")(k), label=f"{model_name} ({source})")
        observed = targets[(targets.target == target) & (targets.zone == "all")].doy.mean()
        ax.axhline(observed, color="k", ls="--", label=f"observed mean {FIRST_YEAR}-{LAST_YEAR}")
        ax.set_xticks(range(len(periods)))
        ax.set_xticklabels([f"{p}\n{GWL_TIMING.get(p, '')}" for p in periods], fontsize=8)
        ax.set_ylabel("day of year (30-year mean per model chain)")
        ax.set_title(f"{target}: projected date for the whole AOI")
        ax.legend(fontsize=8, loc="best")
        ax.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    # Shift per zone (elevation) and warming level, best model of the cross-validation per source
    best = cv_scores[cv_scores.model != "trend"].sort_values("rmse").groupby(["source", "target"]).model.first()
    for (source, target), model_name in best.items():
        p = projections_by_gwl[(projections_by_gwl.source == source) & (projections_by_gwl.target == target)
                               & (projections_by_gwl.model == model_name) & (projections_by_gwl.zone.isin(ZONES))]
        if p.empty or model_name == "mean":
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        for period, g in p.groupby("period"):
            g = g.join(zone_table[["elev_mean", "aspect"]], on="zone").sort_values("elev_mean")
            for aspect, ga in g.groupby("aspect"):
                ax.errorbar(ga.elev_mean, ga.delta_median, yerr=[ga.delta_median - ga.delta_p05, ga.delta_p95 - ga.delta_median],
                            fmt="o-" if aspect == "N" else "s--", capsize=2, alpha=0.8,
                            label=f"{period} {aspect}", color=plt.get_cmap("plasma")(periods.index(period) / len(periods)))
        ax.axhline(0, color="k", lw=0.5)
        ax.set_xlabel("zone mean elevation (m)")
        ax.set_ylabel(f"shift vs {REFERENCE} (days)")
        ax.set_title(f"{target}: {model_name} ({source}), median and 5-95 % of model chains")
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

## 5. Export

In [ ]:
outputs = {
    "climate_features_obs.csv": pd.concat([f.assign(source=s) for s, f in obs_features.items()]),
    "feature_info.csv": feature_info,
    "window_scan.csv": window_scan,
    "climate_effects.csv": climate_effects,
    "process_model_params.csv": process_params,
    "rf_importance.csv": rf_importance,
    "cv_scores.csv": cv_scores,
    "cv_selected_features.csv": chosen,
    "projections_by_gwl.csv": projections_by_gwl,
    "extrapolation.csv": extrapolation,
}
for name, frame in outputs.items():
    frame.to_csv(CLIMATE_OUT_DIR / name, index=False)
print(f"Results written to {CLIMATE_OUT_DIR}")